# 03 — Fetch Game & Season Stats

Loads three tables:
- `game_stats` — per-player, per-game skater stats, pulled from each game's boxscore
- `skater_season_stats` — season totals for skaters
- `goalie_season_stats` — season totals for goalies

**Note:** pulling a boxscore for every completed game can mean a lot of API calls. Start with a small slice (e.g. one team, or `LIMIT 20`) while testing, then run the full loop.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/NHL_Analytics_Hub'
sys.path.append(PROJECT_DIR)
import nhl_api_utils as nhl
import pandas as pd

DB_PATH = f'{PROJECT_DIR}/data/nhl_hockey.db'
conn = nhl.get_connection(DB_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Inspect the full NHL player profile for G. Gawdin

player_id = 8478446

player_profile = nhl.get_json(
    f"{nhl.BASE_WEB}/player/{player_id}/landing"
)

print(type(player_profile))

if player_profile:
    print(player_profile.keys())

#     # Inspect the fields needed to create a players row

# player_id = 8478446

# print("player_id:", player_profile.get("playerId"))
# print("current_team_id:", player_profile.get("currentTeamId"))
# print("current_team_abbrev:", player_profile.get("currentTeamAbbrev"))

# print("first_name:", player_profile.get("firstName"))
# print("last_name:", player_profile.get("lastName"))
# print("position:", player_profile.get("position"))
# print("jersey_number:", player_profile.get("sweaterNumber"))

# print("birth_date:", player_profile.get("birthDate"))
# print("birth_country:", player_profile.get("birthCountry"))

# print("height_cm:", player_profile.get("heightInCentimeters"))
# print("weight_kg:", player_profile.get("weightInKilograms"))
# print("shoots_catches:", player_profile.get("shootsCatches"))
# print("headshot:", player_profile.get("headshot"))

<class 'dict'>
dict_keys(['playerId', 'isActive', 'currentTeamId', 'currentTeamAbbrev', 'fullTeamName', 'teamCommonName', 'teamPlaceNameWithPreposition', 'firstName', 'lastName', 'badges', 'teamLogo', 'sweaterNumber', 'position', 'headshot', 'heroImage', 'heightInInches', 'heightInCentimeters', 'weightInPounds', 'weightInKilograms', 'birthDate', 'birthCity', 'birthStateProvince', 'birthCountry', 'shootsCatches', 'draftDetails', 'playerSlug', 'inTop100AllTime', 'inHHOF', 'featuredStats', 'careerTotals', 'shopLink', 'twitterLink', 'watchLink', 'last5Games', 'seasonTotals', 'currentTeamRoster'])


In [5]:
# Inspect the fields needed to create a players row

player_id = 8478446

print("player_id:", player_profile.get("playerId"))
print("current_team_id:", player_profile.get("currentTeamId"))
print("current_team_abbrev:", player_profile.get("currentTeamAbbrev"))

print("first_name:", player_profile.get("firstName"))
print("last_name:", player_profile.get("lastName"))
print("position:", player_profile.get("position"))
print("jersey_number:", player_profile.get("sweaterNumber"))

print("birth_date:", player_profile.get("birthDate"))
print("birth_country:", player_profile.get("birthCountry"))

print("height_cm:", player_profile.get("heightInCentimeters"))
print("weight_kg:", player_profile.get("weightInKilograms"))
print("shoots_catches:", player_profile.get("shootsCatches"))
print("headshot:", player_profile.get("headshot"))

player_id: 8478446
current_team_id: 3
current_team_abbrev: NYR
first_name: {'default': 'Glenn'}
last_name: {'default': 'Gawdin'}
position: C
jersey_number: 15
birth_date: 1997-03-25
birth_country: CAN
height_cm: 185
weight_kg: 88
shoots_catches: R
headshot: https://assets.nhle.com/mugs/nhl/20262027/NYR/8478446.png


In [6]:
missing_player_ids = [
    8478446,  # Glenn Gawdin
    8484802,  # L. Greentree
    8479366,  # L. Brown
    8479421,  # J. Moverare
    8477473,  # J. Bailey
]

for player_id in missing_player_ids:
    profile = nhl.get_json(
        f"{nhl.BASE_WEB}/player/{player_id}/landing"
    )

    print("=" * 60)

    if not profile:
        print("player_id:", player_id)
        print("PROFILE NOT FOUND")
        continue

    print("player_id:", profile.get("playerId"))
    print(
        "name:",
        profile.get("firstName", {}).get("default"),
        profile.get("lastName", {}).get("default")
    )
    print("current_team:", profile.get("currentTeamAbbrev"))
    print("position:", profile.get("position"))

player_id: 8478446
name: Glenn Gawdin
current_team: NYR
position: C
player_id: 8484802
name: Liam Greentree
current_team: NYR
position: R
player_id: 8479366
name: Logan Brown
current_team: None
position: C
player_id: 8479421
name: Jacob Moverare
current_team: None
position: D
player_id: 8477473
name: Justin Bailey
current_team: None
position: R


In [7]:
player_id = 8478446
team_abbrev = "LAK"

profile = nhl.get_json(
    f"{nhl.BASE_WEB}/player/{player_id}/landing"
)

team_id = nhl.get_team_id(conn, team_abbrev)

player_row = {
    "player_id": profile["playerId"],
    "team_id": team_id,
    "first_name": profile["firstName"]["default"],
    "last_name": profile["lastName"]["default"],
    "position": profile.get("position"),
    "jersey_number": profile.get("sweaterNumber"),
    "birth_date": profile.get("birthDate"),
    "birth_country": profile.get("birthCountry"),
    "height_cm": profile.get("heightInCentimeters"),
    "weight_kg": profile.get("weightInKilograms"),
    "shoots_catches": profile.get("shootsCatches"),
    "headshot_url": profile.get("headshot"),
}

print("Resolved team_id:", team_id)
print("Player row:")
print(player_row)

Resolved team_id: 20
Player row:
{'player_id': 8478446, 'team_id': 20, 'first_name': 'Glenn', 'last_name': 'Gawdin', 'position': 'C', 'jersey_number': 15, 'birth_date': '1997-03-25', 'birth_country': 'CAN', 'height_cm': 185, 'weight_kg': 88, 'shoots_catches': 'R', 'headshot_url': 'https://assets.nhle.com/mugs/nhl/20262027/NYR/8478446.png'}


In [3]:
# Inspect the actual players table definition
players_schema = pd.read_sql_query(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='players';",
    conn
)

print(players_schema.iloc[0]["sql"])

CREATE TABLE players (
    player_id       INTEGER PRIMARY KEY,
    team_id         INTEGER NOT NULL,
    first_name      TEXT,
    last_name       TEXT,
    position        TEXT,
    jersey_number   INTEGER,
    birth_date      TEXT,
    birth_country   TEXT,
    height_cm       REAL,
    weight_kg       REAL,
    shoots_catches  TEXT,
    headshot_url    TEXT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
)


In [6]:
# Test creating a missing historical player parent row

player_id = 8478446
team_abbrev = "LAK"

# Fetch the player's profile
profile = nhl.get_json(
    f"{nhl.BASE_WEB}/player/{player_id}/landing"
)

# Resolve the HISTORICAL team from the boxscore context
team_id = nhl.get_team_id(conn, team_abbrev)

player_row = {
    "player_id": profile["playerId"],
    "team_id": team_id,
    "first_name": profile["firstName"]["default"],
    "last_name": profile["lastName"]["default"],
    "position": profile.get("position"),
    "jersey_number": profile.get("sweaterNumber"),
    "birth_date": profile.get("birthDate"),
    "birth_country": profile.get("birthCountry"),
    "height_cm": profile.get("heightInCentimeters"),
    "weight_kg": profile.get("weightInKilograms"),
    "shoots_catches": profile.get("shootsCatches"),
    "headshot_url": profile.get("headshot"),
}

print("Player row to insert:")
for key, value in player_row.items():
    print(f"{key}: {value}")

# Verify the historical team is valid
print("\nResolved historical team_id:", team_id)

# Check whether the player already exists
existing = conn.execute(
    "SELECT player_id, team_id FROM players WHERE player_id = ?",
    (player_id,)
).fetchone()

print("\nExisting player row:", existing)

Player row to insert:
player_id: 8478446
team_id: 20
first_name: Glenn
last_name: Gawdin
position: C
jersey_number: 15
birth_date: 1997-03-25
birth_country: CAN
height_cm: 185
weight_kg: 92
shoots_catches: R
headshot_url: https://assets.nhle.com/mugs/nhl/20262027/NYR/8478446.png

Resolved historical team_id: 20

Existing player row: None


In [7]:
# Insert the test historical player into the players table

nhl.insert_ignore(conn, "players", player_row)
conn.commit()

print("Player inserted successfully.")

# Verify the exact row now stored in the database
stored_player = conn.execute(
    """
    SELECT
        player_id,
        team_id,
        first_name,
        last_name,
        position,
        jersey_number
    FROM players
    WHERE player_id = ?
    """,
    (player_id,)
).fetchone()

print("\nStored player:")
print(stored_player)

# Verify the team relationship
team_check = conn.execute(
    """
    SELECT
        p.player_id,
        p.first_name,
        p.last_name,
        p.team_id,
        t.team_abbrev
    FROM players p
    JOIN teams t
        ON p.team_id = t.team_id
    WHERE p.player_id = ?
    """,
    (player_id,)
).fetchone()

print("\nPlayer-team relationship:")
print(team_check)

Player inserted successfully.

Stored player:
(8478446, 20, 'Glenn', 'Gawdin', 'C', 15)

Player-team relationship:
(8478446, 'Glenn', 'Gawdin', 20, 'LAK')


In [3]:
#validation test (imp)
season_roster_test = nhl.get_json(
    f"{nhl.BASE_WEB}/roster/LAK/20252026"
)

print(season_roster_test.keys())

for group in ("forwards", "defensemen", "goalies"):
    for p in season_roster_test.get(group, []):
        if p.get("id") == 8478446:
            print("FOUND:", group)
            print(p)

dict_keys(['forwards', 'defensemen', 'goalies'])


In [6]:
for group in ("forwards", "defensemen", "goalies"):
    for p in season_roster_test.get(group, []):
        if p.get("id") == 8478446:
            print("FOUND:", group)
            print(p)

In [7]:
# Validation: inspect the size of the 2025-26 LAK season roster

for group in ("forwards", "defensemen", "goalies"):
    players = season_roster_test.get(group, [])
    print(group, ":", len(players))

total = sum(
    len(season_roster_test.get(group, []))
    for group in ("forwards", "defensemen", "goalies")
)

print("TOTAL:", total)

forwards : 10
defensemen : 6
goalies : 2
TOTAL: 18


In [8]:
# Inspect the players table schema

cursor = conn.execute("PRAGMA table_info(players)")
players_schema = cursor.fetchall()

for column in players_schema:
    print(column)

(0, 'player_id', 'INTEGER', 0, None, 1)
(1, 'team_id', 'INTEGER', 1, None, 0)
(2, 'first_name', 'TEXT', 0, None, 0)
(3, 'last_name', 'TEXT', 0, None, 0)
(4, 'position', 'TEXT', 0, None, 0)
(5, 'jersey_number', 'INTEGER', 0, None, 0)
(6, 'birth_date', 'TEXT', 0, None, 0)
(7, 'birth_country', 'TEXT', 0, None, 0)
(8, 'height_cm', 'REAL', 0, None, 0)
(9, 'weight_kg', 'REAL', 0, None, 0)
(10, 'shoots_catches', 'TEXT', 0, None, 0)
(11, 'headshot_url', 'TEXT', 0, None, 0)


In [ ]:
print(pd.read_sql("""
SELECT game_state, COUNT(*) AS n
FROM games
GROUP BY game_state
ORDER BY game_state
""", conn))

  game_state     n
0      FINAL   104
1        OFF  1394


## Part A — `game_stats` from boxscores

Inspect one boxscore first — the player-stats structure is nested under each team's roster group.

In [16]:
# completed_games = pd.read_sql(
#     "SELECT game_id, home_team_id, away_team_id FROM games WHERE game_state = 'OFF'",
#     conn,
# )

completed_games = pd.read_sql(
    """
    SELECT game_id, home_team_id, away_team_id
    FROM games
    WHERE game_state IN ('OFF', 'FINAL')
    """,
    conn,
)
print(f'{len(completed_games)} completed games found')

sample_box = nhl.fetch_boxscore(int(completed_games.iloc[0]['game_id']))
print(list(sample_box.keys()))
sample_box['playerByGameStats']['homeTeam'].keys()


1498 completed games found
['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'regPeriods', 'awayTeam', 'homeTeam', 'clock', 'playerByGameStats', 'gameOutcome']


dict_keys(['forwards', 'defense', 'goalies'])

In [ ]:
sample_box = nhl.fetch_boxscore(int(completed_games.iloc[0]['game_id']))

print(list(sample_box.keys()))
print(sample_box['playerByGameStats'].keys())
print(sample_box['playerByGameStats']['homeTeam'].keys())
print(sample_box['playerByGameStats']['homeTeam']['forwards'][0])

['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'regPeriods', 'awayTeam', 'homeTeam', 'clock', 'playerByGameStats', 'gameOutcome']
dict_keys(['awayTeam', 'homeTeam'])
dict_keys(['forwards', 'defense', 'goalies'])
{'playerId': 8482155, 'sweaterNumber': 14, 'name': {'default': 'A. Laferriere'}, 'position': 'R', 'goals': 1, 'assists': 0, 'points': 1, 'plusMinus': 1, 'pim': 0, 'hits': 0, 'powerPlayGoals': 0, 'sog': 3, 'blockedShots': 0, 'shifts': 0}


In [ ]:
print("Home:", sample_box['homeTeam']['abbrev'])
print("Away:", sample_box['awayTeam']['abbrev'])

print(
    "Home DB team_id:",
    nhl.get_team_id(conn, sample_box['homeTeam']['abbrev'])
)

print(
    "Away DB team_id:",
    nhl.get_team_id(conn, sample_box['awayTeam']['abbrev'])
)

Home: LAK
Away: ANA
Home DB team_id: 20
Away DB team_id: 18


In [9]:
def ensure_player_exists(conn, player_id, team_abbrev):
    """
    Ensure a player referenced by a historical boxscore exists
    in the players table.

    If the player already exists, do nothing.

    If the player is missing, fetch the player's landing profile
    and create the parent players row using the historical
    boxscore team.
    """

    # Check whether the player already exists
    existing = conn.execute(
        """
        SELECT player_id
        FROM players
        WHERE player_id = ?
        """,
        (player_id,)
    ).fetchone()

    if existing:
        return False

    # Fetch player profile from NHL API
    profile = nhl.get_json(
        f"{nhl.BASE_WEB}/player/{player_id}/landing"
    )

    # Historical team comes from the boxscore, NOT currentTeamId
    team_id = nhl.get_team_id(conn, team_abbrev)

    if team_id is None:
        raise ValueError(
            f"Could not resolve team_id for historical team {team_abbrev}"
        )

    player_row = {
        "player_id": profile["playerId"],
        "team_id": team_id,
        "first_name": profile["firstName"]["default"],
        "last_name": profile["lastName"]["default"],
        "position": profile.get("position"),
        "jersey_number": profile.get("sweaterNumber"),
        "birth_date": profile.get("birthDate"),
        "birth_country": profile.get("birthCountry"),
        "height_cm": profile.get("heightInCentimeters"),
        "weight_kg": profile.get("weightInKilograms"),
        "shoots_catches": profile.get("shootsCatches"),
        "headshot_url": profile.get("headshot"),
    }

    nhl.insert_ignore(conn, "players", player_row)

    return True

In [10]:
result = ensure_player_exists(
    conn,
    player_id=8478446,
    team_abbrev="LAK"
)

print("Player created:", result)

stored = conn.execute(
    """
    SELECT player_id, team_id, first_name, last_name
    FROM players
    WHERE player_id = ?
    """,
    (8478446,)
).fetchone()

print("Stored row:", stored)

Player created: False
Stored row: (8478446, 20, 'Glenn', 'Gawdin')


In [11]:
# Test ensure_player_exists() with a player who is currently missing

test_player_id = 8484802
test_team_abbrev = "LAK"

# Confirm the player is currently absent
before = conn.execute(
    """
    SELECT player_id, team_id, first_name, last_name
    FROM players
    WHERE player_id = ?
    """,
    (test_player_id,)
).fetchone()

print("Before:", before)

# Create the missing player parent row
result = ensure_player_exists(
    conn,
    player_id=test_player_id,
    team_abbrev=test_team_abbrev
)

print("Player created:", result)

# Verify the inserted row
after = conn.execute(
    """
    SELECT player_id, team_id, first_name, last_name
    FROM players
    WHERE player_id = ?
    """,
    (test_player_id,)
).fetchone()

print("After:", after)

Before: None
Player created: True
After: (8484802, 20, 'Liam', 'Greentree')


In [14]:
# def load_boxscore_stats(conn, game_id, box):
#     """Flatten forwards+defense player stat blocks from one boxscore into game_stats rows.
#     Goalies are skipped here per the schema (game_stats covers skaters only)."""
#     stats = box.get('playerByGameStats', {})
#     inserted = 0
#     for side in ('homeTeam', 'awayTeam'):
#         side_stats = stats.get(side, {})
#         team_abbrev = box[side]['abbrev'] if side in box else None
#         team_id = nhl.get_team_id(conn, team_abbrev) if team_abbrev else None
#         for group in ('forwards', 'defense'):
#             for p in side_stats.get(group, []):
#                 row = {
#                     'game_id': game_id,
#                     'player_id': p['playerId'],
#                     'team_id': team_id,
#                     'goals': p.get('goals'),
#                     'assists': p.get('assists'),
#                     'points': p.get('points'),
#                     'shots_on_goal': p.get('sog'),
#                     'penalty_min': p.get('pim'),
#                     'toi': p.get('toi'),
#                     'plus_minus': p.get('plusMinus'),
#                 }
#                 nhl.insert_ignore(conn, 'game_stats', row)
#                 inserted += 1
#     return inserted




def load_boxscore_stats(conn, game_id, box):
    """Flatten forwards+defense player stat blocks from one boxscore
    into game_stats rows.

    Missing historical players are first created in the players table
    using the player's historical boxscore team.

    Goalies are skipped here per the schema
    (game_stats covers skaters only).
    """

    stats = box.get('playerByGameStats', {})
    inserted = 0

    for side in ('homeTeam', 'awayTeam'):
        side_stats = stats.get(side, {})
        team_abbrev = box[side]['abbrev'] if side in box else None
        team_id = nhl.get_team_id(conn, team_abbrev) if team_abbrev else None

        for group in ('forwards', 'defense'):
            for p in side_stats.get(group, []):

                player_id = p['playerId']

                # Make sure the parent player row exists
                # before inserting the game_stats child row.
                ensure_player_exists(
                    conn,
                    player_id=player_id,
                    team_abbrev=team_abbrev
                )

                row = {
                    'game_id': game_id,
                    'player_id': player_id,
                    'team_id': team_id,
                    'goals': p.get('goals'),
                    'assists': p.get('assists'),
                    'points': p.get('points'),
                    'shots_on_goal': p.get('sog'),
                    'penalty_min': p.get('pim'),
                    'toi': p.get('toi'),
                    'plus_minus': p.get('plusMinus'),
                }

                nhl.insert_ignore(conn, 'game_stats', row)
                inserted += 1

    return inserted

In [17]:
#test
# Single-game integration test for game_stats

test_game_id = int(completed_games.iloc[0]['game_id'])

print("Testing game_id:", test_game_id)

# Fetch the boxscore
box = nhl.fetch_boxscore(test_game_id)

print("Boxscore fetched:", box is not None)

# Load game stats
inserted = load_boxscore_stats(
    conn,
    test_game_id,
    box
)

conn.commit()

print("game_stats rows processed:", inserted)

Testing game_id: 2025010001
Boxscore fetched: True
game_stats rows processed: 36


In [18]:
# Validate game_stats rows for the test game

game_stats_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats
    WHERE game_id = ?
    """,
    (test_game_id,)
).fetchone()[0]

print("game_stats rows for test game:", game_stats_count)

# Check for orphan player references
orphan_players = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats gs
    LEFT JOIN players p
        ON gs.player_id = p.player_id
    WHERE gs.game_id = ?
      AND p.player_id IS NULL
    """,
    (test_game_id,)
).fetchone()[0]

print("Orphan player references:", orphan_players)

# Check for orphan team references
orphan_teams = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats gs
    LEFT JOIN teams t
        ON gs.team_id = t.team_id
    WHERE gs.game_id = ?
      AND t.team_id IS NULL
    """,
    (test_game_id,)
).fetchone()[0]

print("Orphan team references:", orphan_teams)

game_stats rows for test game: 36
Orphan player references: 0
Orphan team references: 0


In [19]:
conn.commit()

In [20]:
# Test on a small slice first — change to completed_games to run everything
test_slice = completed_games.head(20)

total_inserted = 0
for _, g in test_slice.iterrows():
    box = nhl.fetch_boxscore(int(g['game_id']))
    if box:
        total_inserted += load_boxscore_stats(conn, int(g['game_id']), box)
conn.commit()
print(f'{total_inserted} game_stats rows inserted (test slice)')


720 game_stats rows inserted (test slice)


In [21]:
# Validate the 20-game game_stats load

test_game_ids = tuple(
    int(game_id)
    for game_id in test_slice['game_id']
)

print("Expected test games:", len(test_game_ids))

# 1. Count distinct games represented in game_stats
distinct_games = conn.execute(
    f"""
    SELECT COUNT(DISTINCT game_id)
    FROM game_stats
    WHERE game_id IN ({','.join(['?'] * len(test_game_ids))})
    """,
    test_game_ids
).fetchone()[0]

print("Distinct games in game_stats:", distinct_games)

# 2. Count total game_stats rows
total_rows = conn.execute(
    f"""
    SELECT COUNT(*)
    FROM game_stats
    WHERE game_id IN ({','.join(['?'] * len(test_game_ids))})
    """,
    test_game_ids
).fetchone()[0]

print("Total game_stats rows:", total_rows)

# 3. Check duplicate (game_id, player_id) pairs
duplicate_pairs = conn.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT game_id, player_id
        FROM game_stats
        WHERE game_id IN ({','.join(['?'] * len(test_game_ids))})
        GROUP BY game_id, player_id
        HAVING COUNT(*) > 1
    )
    """,
    test_game_ids
).fetchone()[0]

print("Duplicate (game_id, player_id) groups:", duplicate_pairs)

# 4. Check orphan players
orphan_players = conn.execute(
    f"""
    SELECT COUNT(*)
    FROM game_stats gs
    LEFT JOIN players p
        ON gs.player_id = p.player_id
    WHERE gs.game_id IN ({','.join(['?'] * len(test_game_ids))})
      AND p.player_id IS NULL
    """,
    test_game_ids
).fetchone()[0]

print("Orphan player references:", orphan_players)

# 5. Check orphan teams
orphan_teams = conn.execute(
    f"""
    SELECT COUNT(*)
    FROM game_stats gs
    LEFT JOIN teams t
        ON gs.team_id = t.team_id
    WHERE gs.game_id IN ({','.join(['?'] * len(test_game_ids))})
      AND t.team_id IS NULL
    """,
    test_game_ids
).fetchone()[0]

print("Orphan team references:", orphan_teams)

Expected test games: 20
Distinct games in game_stats: 20
Total game_stats rows: 720
Duplicate (game_id, player_id) groups: 0
Orphan player references: 0
Orphan team references: 0


In [22]:
# Inspect players created/used by the 20-game test

test_player_count = conn.execute(
    f"""
    SELECT COUNT(DISTINCT player_id)
    FROM game_stats
    WHERE game_id IN ({','.join(['?'] * len(test_game_ids))})
    """,
    test_game_ids
).fetchone()[0]

print("Distinct players in 20-game game_stats:", test_player_count)

# Show players referenced by game_stats whose player rows were
# created with a historical team relationship.
sample_players = pd.read_sql_query(
    f"""
    SELECT DISTINCT
        gs.player_id,
        p.first_name,
        p.last_name,
        p.team_id,
        t.team_abbrev
    FROM game_stats gs
    JOIN players p
        ON gs.player_id = p.player_id
    JOIN teams t
        ON p.team_id = t.team_id
    WHERE gs.game_id IN ({','.join(['?'] * len(test_game_ids))})
    ORDER BY p.last_name, p.first_name
    LIMIT 20
    """,
    conn,
    params=test_game_ids
)

print("\nSample player relationships:")
print(sample_players.to_string(index=False))

Distinct players in 20-game game_stats: 685

Sample player relationships:
 player_id first_name      last_name  team_id team_abbrev
   8479022    Rodrigo          Abols       11         PHI
   8480884      Calen        Addison       21         NJD
   8481585       Egor      Afanasyev       24         SJS
   8475461     Andrew       Agozzino       15         UTA
   8483390      Roman          Ahcan       17         CBJ
   8485352    Kashawn      Aitcheson       19         NYI
   8484139       Beau           Akey       14         EDM
   8481543     Nikita     Alexandrov       22         STL
   8483687      Marek        Alscher       25         FLA
   8482496       Nils           Aman       32         VAN
   8485589      Lirim      Amidovski        7         MIN
   8479315       Joey       Anderson       31         CHI
   8476981       Josh       Anderson        6         MTL
   8479994      Jaret Anderson-Dolan       26         WPG
   8478397     Rasmus      Andersson       13         VG

In [23]:
# Validate historical team relationships in game_stats

historical_team_check = pd.read_sql_query(
    f"""
    SELECT
        gs.player_id,
        p.first_name,
        p.last_name,
        p.team_id AS player_team_id,
        pt.team_abbrev AS player_team,
        gs.team_id AS game_team_id,
        gt.team_abbrev AS game_team,
        COUNT(*) AS games_played
    FROM game_stats gs
    JOIN players p
        ON gs.player_id = p.player_id
    JOIN teams pt
        ON p.team_id = pt.team_id
    JOIN teams gt
        ON gs.team_id = gt.team_id
    WHERE gs.game_id IN ({','.join(['?'] * len(test_game_ids))})
    GROUP BY
        gs.player_id,
        gs.team_id
    ORDER BY
        p.last_name,
        p.first_name
    LIMIT 30
    """,
    conn,
    params=test_game_ids
)

print(historical_team_check.to_string(index=False))

 player_id first_name      last_name  player_team_id player_team  game_team_id game_team  games_played
   8479022    Rodrigo          Abols              11         PHI            11       PHI             1
   8480884      Calen        Addison              21         NJD            21       NJD             1
   8481585       Egor      Afanasyev              24         SJS            24       SJS             1
   8475461     Andrew       Agozzino              15         UTA            15       UTA             1
   8483390      Roman          Ahcan              17         CBJ            17       CBJ             2
   8485352    Kashawn      Aitcheson              19         NYI            19       NYI             1
   8484139       Beau           Akey              14         EDM            14       EDM             1
   8481543     Nikita     Alexandrov              22         STL            22       STL             1
   8483687      Marek        Alscher              25         FLA         

In [24]:
# Inspect the types of completed games before full game_stats loading

game_type_distribution = pd.read_sql_query(
    """
    SELECT
        game_type,
        COUNT(*) AS game_count
    FROM games
    WHERE game_state IN ('OFF', 'FINAL')
    GROUP BY game_type
    ORDER BY game_type
    """,
    conn
)

print(game_type_distribution.to_string(index=False))

 game_type  game_count
         1         104
         2        1312
         3          82


In [25]:
# Inspect representative completed games for each game_type

game_type_samples = pd.read_sql_query(
    """
    SELECT
        game_id,
        season,
        game_type,
        game_date,
        home_team_id,
        away_team_id,
        game_state
    FROM games
    WHERE game_state IN ('OFF', 'FINAL')
    ORDER BY game_type, game_date
    LIMIT 30
    """,
    conn
)

print(game_type_samples.to_string(index=False))

   game_id   season  game_type  game_date  home_team_id  away_team_id game_state
2025010101 20252026          1 2025-09-20             3            22      FINAL
2025010001 20252026          1 2025-09-21            20            18      FINAL
2025010002 20252026          1 2025-09-21            14            29      FINAL
2025010003 20252026          1 2025-09-21            29            14      FINAL
2025010004 20252026          1 2025-09-21            23            25      FINAL
2025010005 20252026          1 2025-09-21            23            25      FINAL
2025010006 20252026          1 2025-09-21            26             7      FINAL
2025010007 20252026          1 2025-09-21            21            30      FINAL
2025010008 20252026          1 2025-09-21            19            11      FINAL
2025010009 20252026          1 2025-09-21            17            22      FINAL
2025010010 20252026          1 2025-09-21             9            28      FINAL
2025010011 20252026         

In [26]:
# Inspect the first regular-season games

regular_season_games = pd.read_sql_query(
    """
    SELECT
        game_id,
        season,
        game_type,
        game_date,
        home_team_id,
        away_team_id,
        game_state
    FROM games
    WHERE game_state IN ('OFF', 'FINAL')
      AND game_type = 2
    ORDER BY game_date, game_id
    LIMIT 10
    """,
    conn
)

print(regular_season_games.to_string(index=False))

   game_id   season  game_type  game_date  home_team_id  away_team_id game_state
2025020001 20252026          2 2025-10-07            25            31        OFF
2025020002 20252026          2 2025-10-07            30            10        OFF
2025020003 20252026          2 2025-10-07            20             1        OFF
2025020004 20252026          2 2025-10-08            28             6        OFF
2025020005 20252026          2 2025-10-08            12             8        OFF
2025020006 20252026          2 2025-10-08            14            29        OFF
2025020007 20252026          2 2025-10-08            13            20        OFF
2025020008 20252026          2 2025-10-09             8            31        OFF
2025020009 20252026          2 2025-10-09             4            30        OFF
2025020010 20252026          2 2025-10-09            16             6        OFF


In [27]:
# Test game_stats loading for the first regular-season game

regular_game_id = int(regular_season_games.iloc[0]['game_id'])

print("Testing regular-season game:", regular_game_id)

box = nhl.fetch_boxscore(regular_game_id)

print("Boxscore fetched:", box is not None)

if box:
    before = conn.execute(
        """
        SELECT COUNT(*)
        FROM game_stats
        WHERE game_id = ?
        """,
        (regular_game_id,)
    ).fetchone()[0]

    print("Rows before load:", before)

    processed = load_boxscore_stats(
        conn,
        regular_game_id,
        box
    )

    conn.commit()

    after = conn.execute(
        """
        SELECT COUNT(*)
        FROM game_stats
        WHERE game_id = ?
        """,
        (regular_game_id,)
    ).fetchone()[0]

    print("Rows processed:", processed)
    print("Rows after load:", after)

Testing regular-season game: 2025020001
Boxscore fetched: True
Rows before load: 0
Rows processed: 36
Rows after load: 36


In [28]:
# Current game_stats status before the full load

total_game_stats = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats
    """
).fetchone()[0]

loaded_games = conn.execute(
    """
    SELECT COUNT(DISTINCT game_id)
    FROM game_stats
    """
).fetchone()[0]

print("Total game_stats rows:", total_game_stats)
print("Games with game_stats:", loaded_games)

Total game_stats rows: 756
Games with game_stats: 21


In [29]:
# Find completed games that do not yet have game_stats

games_pending_stats = pd.read_sql_query(
    """
    SELECT
        g.game_id,
        g.season,
        g.game_type,
        g.game_date,
        g.home_team_id,
        g.away_team_id
    FROM games g
    LEFT JOIN game_stats gs
        ON g.game_id = gs.game_id
    WHERE g.game_state IN ('OFF', 'FINAL')
      AND gs.game_id IS NULL
    ORDER BY g.game_date, g.game_id
    """,
    conn
)

print("Games pending game_stats:", len(games_pending_stats))

print("\nPending games by game_type:")
print(
    games_pending_stats['game_type']
    .value_counts()
    .sort_index()
    .to_string()
)

Games pending game_stats: 1477

Pending games by game_type:
game_type
1      84
2    1311
3      82


In [30]:
# Full game_stats loader
#
# Loads only completed games that do not yet have game_stats.
# Commits in batches so the notebook can be safely resumed.

import time

BATCH_SIZE = 25

loaded_games = 0
failed_games = []
processed_rows = 0

pending_games = games_pending_stats.copy()

print(f"Games to process: {len(pending_games)}")
print(f"Batch size: {BATCH_SIZE}")

for index, game in pending_games.iterrows():

    game_id = int(game["game_id"])

    try:
        box = nhl.fetch_boxscore(game_id)

        if not box:
            failed_games.append({
                "game_id": game_id,
                "reason": "Boxscore returned no data"
            })
            continue

        rows_processed = load_boxscore_stats(
            conn,
            game_id,
            box
        )

        processed_rows += rows_processed
        loaded_games += 1

        # Commit periodically
        if loaded_games % BATCH_SIZE == 0:
            conn.commit()

            print(
                f"Progress: {loaded_games}/{len(pending_games)} "
                f"games processed | "
                f"{processed_rows} player-stat records processed"
            )

        # Small pause between API requests
        time.sleep(0.05)

    except Exception as e:

        failed_games.append({
            "game_id": game_id,
            "reason": str(e)
        })

        print(
            f"FAILED game {game_id}: {type(e).__name__}: {e}"
        )

# Commit any remaining successful work
conn.commit()

print("\nFull game_stats load finished.")
print("Games processed successfully:", loaded_games)
print("Player-stat records processed:", processed_rows)
print("Failed games:", len(failed_games))

Games to process: 1477
Batch size: 25
Progress: 25/1477 games processed | 900 player-stat records processed
Progress: 50/1477 games processed | 1800 player-stat records processed
Progress: 75/1477 games processed | 2700 player-stat records processed
Progress: 100/1477 games processed | 3600 player-stat records processed
Progress: 125/1477 games processed | 4500 player-stat records processed
Progress: 150/1477 games processed | 5400 player-stat records processed
Progress: 175/1477 games processed | 6300 player-stat records processed
Progress: 200/1477 games processed | 7200 player-stat records processed
Progress: 225/1477 games processed | 8100 player-stat records processed
Progress: 250/1477 games processed | 9000 player-stat records processed
Progress: 275/1477 games processed | 9900 player-stat records processed
Progress: 300/1477 games processed | 10800 player-stat records processed
Progress: 325/1477 games processed | 11700 player-stat records processed
Progress: 350/1477 games pro

In [31]:
# Full game_stats validation

total_games = conn.execute(
    """
    SELECT COUNT(*)
    FROM games
    WHERE game_state IN ('OFF', 'FINAL')
    """
).fetchone()[0]

games_with_stats = conn.execute(
    """
    SELECT COUNT(DISTINCT game_id)
    FROM game_stats
    """
).fetchone()[0]

total_stats_rows = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats
    """
).fetchone()[0]

duplicate_pairs = conn.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT game_id, player_id
        FROM game_stats
        GROUP BY game_id, player_id
        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]

orphan_players = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats gs
    LEFT JOIN players p
        ON gs.player_id = p.player_id
    WHERE p.player_id IS NULL
    """
).fetchone()[0]

orphan_teams = conn.execute(
    """
    SELECT COUNT(*)
    FROM game_stats gs
    LEFT JOIN teams t
        ON gs.team_id = t.team_id
    WHERE t.team_id IS NULL
    """
).fetchone()[0]

print("Completed games:", total_games)
print("Games with game_stats:", games_with_stats)
print("Total game_stats rows:", total_stats_rows)
print("Duplicate (game_id, player_id) groups:", duplicate_pairs)
print("Orphan player references:", orphan_players)
print("Orphan team references:", orphan_teams)

Completed games: 1498
Games with game_stats: 1498
Total game_stats rows: 53927
Duplicate (game_id, player_id) groups: 0
Orphan player references: 0
Orphan team references: 0


In [32]:
# Validate game_stats coverage by game type

game_type_stats = pd.read_sql_query(
    """
    SELECT
        g.game_type,
        COUNT(DISTINCT g.game_id) AS games,
        COUNT(gs.player_id) AS game_stats_rows
    FROM games g
    LEFT JOIN game_stats gs
        ON g.game_id = gs.game_id
    WHERE g.game_state IN ('OFF', 'FINAL')
    GROUP BY g.game_type
    ORDER BY g.game_type
    """,
    conn
)

print(game_type_stats.to_string(index=False))

 game_type  games  game_stats_rows
         1    104             3744
         2   1312            47231
         3     82             2952


In [33]:
# Check how the players table changed after historical boxscore loading

player_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM players
    """
).fetchone()[0]

print("Total players:", player_count)

Total players: 1669


In [34]:
# Inspect one page of skater season-summary data

skater_sample = nhl.fetch_skater_summary(
    start=0,
    limit=5
)

print("Top-level keys:")
print(skater_sample.keys())

print("\nNumber of records:")
print(len(skater_sample.get("data", [])))

if skater_sample.get("data"):
    print("\nFirst skater record:")
    print(skater_sample["data"][0])

Top-level keys:
dict_keys(['data', 'total'])

Number of records:
5

First skater record:
{'assists': 90, 'evGoals': 34, 'evPoints': 82, 'faceoffWinPct': 0.49512, 'gameWinningGoals': 4, 'gamesPlayed': 82, 'goals': 48, 'lastName': 'McDavid', 'otGoals': 1, 'penaltyMinutes': 44, 'playerId': 8478402, 'plusMinus': 17, 'points': 138, 'pointsPerGame': 1.68292, 'positionCode': 'C', 'ppGoals': 13, 'ppPoints': 54, 'seasonId': 20252026, 'shGoals': 1, 'shPoints': 2, 'shootingPct': 0.15686, 'shootsCatches': 'L', 'shots': 306, 'skaterFullName': 'Connor McDavid', 'teamAbbrevs': 'EDM', 'timeOnIcePerGame': 1379.1219}


In [35]:
# Inspect player/team/season combinations in the skater summary

skater_sample = nhl.fetch_skater_summary(
    start=0,
    limit=100
)

skater_df = pd.DataFrame(
    skater_sample.get("data", [])
)

print("Shape:", skater_df.shape)

print("\nRelevant columns:")
print(
    [
        c for c in [
            "playerId",
            "seasonId",
            "teamAbbrevs",
            "gamesPlayed",
            "goals",
            "assists",
            "points",
            "plusMinus",
            "penaltyMinutes",
            "shots",
            "timeOnIcePerGame"
        ]
        if c in skater_df.columns
    ]
)

if "teamAbbrevs" in skater_df.columns:
    print("\nSample player/team/season values:")
    print(
        skater_df[
            ["playerId", "seasonId", "teamAbbrevs"]
        ].head(20).to_string(index=False)
    )

Shape: (100, 26)

Relevant columns:
['playerId', 'seasonId', 'teamAbbrevs', 'gamesPlayed', 'goals', 'assists', 'points', 'plusMinus', 'penaltyMinutes', 'shots', 'timeOnIcePerGame']

Sample player/team/season values:
 playerId  seasonId teamAbbrevs
  8478402  20252026         EDM
  8476453  20252026         TBL
  8477492  20252026         COL
  8484801  20252026         SJS
  8476460  20252026         WPG
  8480018  20252026         MTL
  8477956  20252026         BOS
  8480039  20252026         COL
  8477934  20252026         EDM
  8480027  20252026         DAL
  8480803  20252026         EDM
  8478398  20252026         WPG
  8478403  20252026         VGK
  8478864  20252026         MIN
  8479343  20252026         UTA
  8481540  20252026         MTL
  8477404  20252026         TBL
  8482740  20252026         DAL
  8481557  20252026         MIN
  8479337  20252026         DET


In [36]:
# Check whether the skater API returns multiple rows
# for the same player in the same season.

duplicate_player_seasons = (
    skater_df
    .groupby(["playerId", "seasonId"])
    .size()
    .reset_index(name="row_count")
)

duplicate_player_seasons = duplicate_player_seasons[
    duplicate_player_seasons["row_count"] > 1
]

print(
    "Duplicate player-season combinations in sample:",
    len(duplicate_player_seasons)
)

if len(duplicate_player_seasons) > 0:
    print(
        duplicate_player_seasons.to_string(index=False)
    )
else:
    print("No duplicate player-season combinations found in this sample.")

Duplicate player-season combinations in sample: 0
No duplicate player-season combinations found in this sample.


In [37]:
# Check how many skater season-stat records the API reports.

print("Total skater season-stat records:", skater_sample.get("total"))

Total skater season-stat records: 940


In [38]:
# Check whether the team abbreviations returned by the API
# can all be resolved to our teams table.

team_abbrevs = (
    skater_df["teamAbbrevs"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Unique team abbreviations in sample:", len(team_abbrevs))
print(sorted(team_abbrevs))

missing_teams = []

for abbrev in team_abbrevs:
    team_id = nhl.get_team_id(conn, abbrev)

    if team_id is None:
        missing_teams.append(abbrev)

print("\nTeam abbreviations not found in teams table:")
print(missing_teams)

Unique team abbreviations in sample: 32
['ANA', 'BOS', 'BUF', 'CAR', 'CBJ', 'CHI', 'COL', 'DAL', 'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NJD', 'NSH', 'NYI', 'NYR', 'NYR,LAK', 'OTT', 'PHI', 'PIT', 'SJS', 'STL', 'TBL', 'TOR', 'UTA', 'VAN,MIN', 'VGK', 'WPG', 'WSH', 'WSH,ANA']

Team abbreviations not found in teams table:
['NYR,LAK', 'VAN,MIN', 'WSH,ANA']


In [39]:
# Fetch all skater season-stat records for validation only.
# This does NOT write anything to the database.

all_skater_records = []

start = 0
limit = 100

while True:
    page = nhl.fetch_skater_summary(start=start, limit=limit)

    if not page or not page.get("data"):
        break

    all_skater_records.extend(page["data"])

    if len(page["data"]) < limit:
        break

    start += limit

print("Records fetched:", len(all_skater_records))

all_skater_df = pd.DataFrame(all_skater_records)

print("\nDataFrame shape:", all_skater_df.shape)

# Check player-season duplicates across the FULL API result
full_duplicates = (
    all_skater_df
    .groupby(["playerId", "seasonId"])
    .size()
    .reset_index(name="row_count")
)

full_duplicates = full_duplicates[
    full_duplicates["row_count"] > 1
]

print("\nDuplicate player-season combinations:", len(full_duplicates))

if len(full_duplicates) > 0:
    print(full_duplicates.to_string(index=False))

# Show players whose teamAbbrevs contains multiple teams
multi_team_skater_df = all_skater_df[
    all_skater_df["teamAbbrevs"]
    .fillna("")
    .str.contains(",", regex=False)
]

print("\nMulti-team player-season records:", len(multi_team_skater_df))

if len(multi_team_skater_df) > 0:
    print(
        multi_team_skater_df[
            [
                "playerId",
                "skaterFullName",
                "seasonId",
                "teamAbbrevs",
                "gamesPlayed",
                "goals",
                "assists",
                "points"
            ]
        ].to_string(index=False)
    )

Records fetched: 940

DataFrame shape: (940, 26)

Duplicate player-season combinations: 0

Multi-team player-season records: 81
 playerId        skaterFullName  seasonId teamAbbrevs  gamesPlayed  goals  assists  points
  8478550        Artemi Panarin  20252026     NYR,LAK           78     28       56      84
  8480800          Quinn Hughes  20252026     VAN,MIN           74      7       69      76
  8474590          John Carlson  20252026     WSH,ANA           71     14       46      60
  8475172           Nazem Kadri  20252026     CGY,COL           77     16       34      50
  8478397      Rasmus Andersson  20252026     CGY,VGK           81     17       30      47
  8482259          Bobby McMann  20252026     TOR,SEA           78     29       17      46
  8478975       Mason Marchment  20252026     SEA,CBJ           68     19       26      45
  8482475        Egor Chinakhov  20252026     CBJ,PIT           72     21       21      42
  8475753          Justin Faulk  20252026     STL,DET

In [40]:
print("skater_season_stats schema:\n")

schema_rows = conn.execute(
    "PRAGMA table_info(skater_season_stats)"
).fetchall()

for row in schema_rows:
    print(row)

print("\nIndexes:\n")

index_rows = conn.execute(
    "PRAGMA index_list(skater_season_stats)"
).fetchall()

for row in index_rows:
    print(row)

skater_season_stats schema:

(0, 'stat_id', 'INTEGER', 0, None, 1)
(1, 'player_id', 'INTEGER', 1, None, 0)
(2, 'season', 'TEXT', 0, None, 0)
(3, 'team_id', 'INTEGER', 0, None, 0)
(4, 'games_played', 'INTEGER', 0, None, 0)
(5, 'goals', 'INTEGER', 0, None, 0)
(6, 'assists', 'INTEGER', 0, None, 0)
(7, 'points', 'INTEGER', 0, None, 0)
(8, 'plus_minus', 'INTEGER', 0, None, 0)
(9, 'penalty_min', 'INTEGER', 0, None, 0)
(10, 'shots', 'INTEGER', 0, None, 0)
(11, 'avg_toi', 'TEXT', 0, None, 0)

Indexes:

(0, 'sqlite_autoindex_skater_season_stats_1', 1, 'u', 0)


In [41]:
print("Unique index columns:\n")

index_info = conn.execute(
    "PRAGMA index_info('sqlite_autoindex_skater_season_stats_1')"
).fetchall()

for row in index_info:
    print(row)

Unique index columns:

(0, 1, 'player_id')
(1, 2, 'season')


In [42]:
# Validate that every skater-season API player exists
# in the players table before insertion.

api_player_ids = set(
    all_skater_df["playerId"]
    .dropna()
    .astype(int)
)

missing_player_ids = []

for player_id in sorted(api_player_ids):
    exists = conn.execute(
        """
        SELECT 1
        FROM players
        WHERE player_id = ?
        """,
        (player_id,)
    ).fetchone()

    if not exists:
        missing_player_ids.append(player_id)

print("Unique API players:", len(api_player_ids))
print("Players missing from players table:", len(missing_player_ids))

if missing_player_ids:
    print("\nMissing player IDs:")
    print(missing_player_ids)
else:
    print("All API skater players exist in players table.")

Unique API players: 940
Players missing from players table: 0
All API skater players exist in players table.


In [ ]:
conn.rollback()

print("Rollback completed.")

Rollback completed.


In [ ]:
for _, g in test_slice.iterrows():
    game_id = int(g['game_id'])
    box = nhl.fetch_boxscore(game_id)

    if not box:
        continue

    stats = box.get('playerByGameStats', {})

    for side in ('homeTeam', 'awayTeam'):
        side_stats = stats.get(side, {})

        team_abbrev = box[side]['abbrev']
        team_id = nhl.get_team_id(conn, team_abbrev)

        print(f"\nGame: {game_id}")
        print(f"{side}: {team_abbrev} -> team_id={team_id}")

        for group in ('forwards', 'defense'):
            for p in side_stats.get(group, []):
                player_id = p['playerId']

                player_exists = conn.execute(
                    "SELECT 1 FROM players WHERE player_id = ?",
                    (player_id,)
                ).fetchone()

                game_exists = conn.execute(
                    "SELECT 1 FROM games WHERE game_id = ?",
                    (game_id,)
                ).fetchone()

                team_exists = conn.execute(
                    "SELECT 1 FROM teams WHERE team_id = ?",
                    (team_id,)
                ).fetchone()

                if not player_exists or not game_exists or not team_exists:
                    print("FOREIGN KEY PROBLEM FOUND")
                    print("  game_id:", game_id, "exists:", bool(game_exists))
                    print("  player_id:", player_id, "exists:", bool(player_exists))
                    print("  team_id:", team_id, "exists:", bool(team_exists))
                    print("  team_abbrev:", team_abbrev)
                    print("  player:", p.get('name', {}).get('default'))
                    print("  group:", group)


Game: 2025010001
homeTeam: LAK -> team_id=20
FOREIGN KEY PROBLEM FOUND
  game_id: 2025010001 exists: True
  player_id: 8478446 exists: False
  team_id: 20 exists: True
  team_abbrev: LAK
  player: G. Gawdin
  group: forwards
FOREIGN KEY PROBLEM FOUND
  game_id: 2025010001 exists: True
  player_id: 8484802 exists: False
  team_id: 20 exists: True
  team_abbrev: LAK
  player: L. Greentree
  group: forwards
FOREIGN KEY PROBLEM FOUND
  game_id: 2025010001 exists: True
  player_id: 8479366 exists: False
  team_id: 20 exists: True
  team_abbrev: LAK
  player: L. Brown
  group: forwards
FOREIGN KEY PROBLEM FOUND
  game_id: 2025010001 exists: True
  player_id: 8479421 exists: False
  team_id: 20 exists: True
  team_abbrev: LAK
  player: J. Moverare
  group: defense

Game: 2025010001
awayTeam: ANA -> team_id=18
FOREIGN KEY PROBLEM FOUND
  game_id: 2025010001 exists: True
  player_id: 8477473 exists: False
  team_id: 18 exists: True
  team_abbrev: ANA
  player: J. Bailey
  group: forwards

Game

Once you've confirmed the test slice worked, swap `test_slice` for `completed_games` and re-run the loop above to load every game.

## Part B — `skater_season_stats`

In [ ]:
sample_sk = nhl.fetch_skater_summary(start=0, limit=5)
print(sample_sk['total'])
sample_sk['data'][0]


In [43]:
# start, limit = 0, 100
# sk_loaded = 0

# while True:
#     page = nhl.fetch_skater_summary(start=start, limit=limit)
#     if not page or not page.get('data'):
#         break
#     for p in page['data']:
#         team_abbrev = (p.get('teamAbbrevs') or '').split(',')[0] or None
#         team_id = nhl.get_team_id(conn, team_abbrev) if team_abbrev else None
#         row = {
#             'player_id': p['playerId'],
#             'season': str(p.get('seasonId')),
#             'team_id': team_id,
#             'games_played': p.get('gamesPlayed'),
#             'goals': p.get('goals'),
#             'assists': p.get('assists'),
#             'points': p.get('points'),
#             'plus_minus': p.get('plusMinus'),
#             'penalty_min': p.get('penaltyMinutes'),
#             'shots': p.get('shots'),
#             'avg_toi': p.get('timeOnIcePerGame'),
#         }
#         nhl.insert_ignore(conn, 'skater_season_stats', row)
#         sk_loaded += 1
#     if len(page['data']) < limit:
#         break
#     start += limit

# conn.commit()
# print(f'{sk_loaded} skater_season_stats rows inserted')



# ============================================================
# Load skater season statistics
# One row per (player_id, season)
# ============================================================

import time

BATCH_SIZE = 100
start = 0
limit = 100

sk_loaded = 0
sk_failed = []

while True:

    page = nhl.fetch_skater_summary(
        start=start,
        limit=limit
    )

    if not page or not page.get("data"):
        break

    for p in page["data"]:

        try:
            player_id = p["playerId"]
            season = str(p["seasonId"])

            # The API can return multiple teams for a player
            # during one season, e.g. "EDM,PIT,COL".
            #
            # The existing schema has only one team_id column,
            # so we retain the first listed team as the
            # representative team for this season-total row.
            team_abbrevs = (p.get("teamAbbrevs") or "").strip()

            team_abbrev = (
                team_abbrevs.split(",")[0].strip()
                if team_abbrevs
                else None
            )

            team_id = (
                nhl.get_team_id(conn, team_abbrev)
                if team_abbrev
                else None
            )

            if team_abbrev and team_id is None:
                raise ValueError(
                    f"Team abbreviation {team_abbrev} "
                    f"not found for player {player_id}"
                )

            row = {
                "player_id": player_id,
                "season": season,
                "team_id": team_id,
                "games_played": p.get("gamesPlayed"),
                "goals": p.get("goals"),
                "assists": p.get("assists"),
                "points": p.get("points"),
                "plus_minus": p.get("plusMinus"),
                "penalty_min": p.get("penaltyMinutes"),
                "shots": p.get("shots"),
                "avg_toi": p.get("timeOnIcePerGame"),
            }

            nhl.insert_ignore(
                conn,
                "skater_season_stats",
                row
            )

            sk_loaded += 1

        except Exception as e:

            sk_failed.append({
                "player_id": p.get("playerId"),
                "season": p.get("seasonId"),
                "reason": str(e)
            })

            print(
                f"FAILED player {p.get('playerId')}: "
                f"{type(e).__name__}: {e}"
            )

    # Commit after every page
    conn.commit()

    print(
        f"Progress: {sk_loaded}/{page.get('total', '?')} "
        f"skater records processed"
    )

    if len(page["data"]) < limit:
        break

    start += limit

    time.sleep(0.05)


print("\nSkater season-stat load finished.")
print("Records processed:", sk_loaded)
print("Failed records:", len(sk_failed))

Progress: 100/940 skater records processed
Progress: 200/940 skater records processed
Progress: 300/940 skater records processed
Progress: 400/940 skater records processed
Progress: 500/940 skater records processed
Progress: 600/940 skater records processed
Progress: 700/940 skater records processed
Progress: 800/940 skater records processed
Progress: 900/940 skater records processed
Progress: 940/940 skater records processed

Skater season-stat load finished.
Records processed: 940
Failed records: 0


In [44]:
print(
    pd.read_sql(
        "SELECT COUNT(*) AS n FROM skater_season_stats",
        conn
    )
)

     n
0  940


In [45]:
# ============================================================
# Goalie season-stat API validation
# Read-only: does NOT write to the database
# ============================================================

sample_g = nhl.fetch_goalie_summary(
    start=0,
    limit=100
)

print("Top-level keys:")
print(sample_g.keys())

print("\nTotal goalie season-stat records:")
print(sample_g.get("total"))

goalie_df = pd.DataFrame(sample_g.get("data", []))

print("\nSample shape:")
print(goalie_df.shape)

print("\nColumns:")
print(goalie_df.columns.tolist())

print("\nFirst goalie record:")
print(sample_g["data"][0])

Top-level keys:
dict_keys(['data', 'total'])

Total goalie season-stat records:
98

Sample shape:
(98, 23)

Columns:
['assists', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'goals', 'goalsAgainst', 'goalsAgainstAverage', 'lastName', 'losses', 'otLosses', 'penaltyMinutes', 'playerId', 'points', 'savePct', 'saves', 'seasonId', 'shootsCatches', 'shotsAgainst', 'shutouts', 'teamAbbrevs', 'ties', 'timeOnIce', 'wins']

First goalie record:
{'assists': 2, 'gamesPlayed': 58, 'gamesStarted': 58, 'goalieFullName': 'Andrei Vasilevskiy', 'goals': 0, 'goalsAgainst': 132, 'goalsAgainstAverage': 2.30853, 'lastName': 'Vasilevskiy', 'losses': 15, 'otLosses': 4, 'penaltyMinutes': 11, 'playerId': 8476883, 'points': 2, 'savePct': 0.91233, 'saves': 1353, 'seasonId': 20252026, 'shootsCatches': 'L', 'shotsAgainst': 1483, 'shutouts': 2, 'teamAbbrevs': 'TBL', 'ties': None, 'timeOnIce': 205845, 'wins': 39}


In [46]:
# ============================================================
# Full goalie season-stat validation
# Read-only: does NOT write to the database
# ============================================================

all_goalie_records = []

start = 0
limit = 100

while True:

    page = nhl.fetch_goalie_summary(
        start=start,
        limit=limit
    )

    if not page or not page.get("data"):
        break

    all_goalie_records.extend(page["data"])

    if len(page["data"]) < limit:
        break

    start += limit


all_goalie_df = pd.DataFrame(all_goalie_records)

print("Records fetched:", len(all_goalie_records))
print("DataFrame shape:", all_goalie_df.shape)


# ------------------------------------------------------------
# Check duplicate player-season combinations
# ------------------------------------------------------------

goalie_duplicates = (
    all_goalie_df
    .groupby(["playerId", "seasonId"])
    .size()
    .reset_index(name="row_count")
)

goalie_duplicates = goalie_duplicates[
    goalie_duplicates["row_count"] > 1
]

print(
    "\nDuplicate player-season combinations:",
    len(goalie_duplicates)
)

if len(goalie_duplicates) > 0:
    print(goalie_duplicates.to_string(index=False))
else:
    print("No duplicate player-season combinations found.")


# ------------------------------------------------------------
# Check multi-team goalies
# ------------------------------------------------------------

multi_team_goalies = all_goalie_df[
    all_goalie_df["teamAbbrevs"]
    .fillna("")
    .str.contains(",", regex=False)
]

print(
    "\nMulti-team goalie-season records:",
    len(multi_team_goalies)
)

if len(multi_team_goalies) > 0:
    print(
        multi_team_goalies[
            [
                "playerId",
                "goalieFullName",
                "seasonId",
                "teamAbbrevs",
                "gamesPlayed",
                "wins",
                "losses",
                "savePct"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Check all goalie players exist in players table
# ------------------------------------------------------------

goalie_player_ids = set(
    all_goalie_df["playerId"]
    .dropna()
    .astype(int)
)

missing_goalie_players = []

for player_id in sorted(goalie_player_ids):

    exists = conn.execute(
        """
        SELECT 1
        FROM players
        WHERE player_id = ?
        """,
        (player_id,)
    ).fetchone()

    if not exists:
        missing_goalie_players.append(player_id)


print(
    "\nUnique goalie players:",
    len(goalie_player_ids)
)

print(
    "Goalie players missing from players table:",
    len(missing_goalie_players)
)

if missing_goalie_players:
    print("\nMissing player IDs:")
    print(missing_goalie_players)
else:
    print("All goalie players exist in players table.")

Records fetched: 98
DataFrame shape: (98, 23)

Duplicate player-season combinations: 0
No duplicate player-season combinations found.

Multi-team goalie-season records: 2
 playerId goalieFullName  seasonId teamAbbrevs  gamesPlayed  wins  losses  savePct
  8479973 Stuart Skinner  20252026     EDM,PIT           50    23      17  0.88792
  8477465  Tristan Jarry  20252026     PIT,EDM           33    18       9  0.88221

Unique goalie players: 98
Goalie players missing from players table: 11

Missing player IDs:
[8471734, 8473503, 8475852, 8476899, 8477405, 8477484, 8478971, 8482411, 8483668, 8483746, 8484910]


In [47]:
# ============================================================
# Ensure all goalie season-stat players exist
# Read-only API lookup + INSERT into players for missing rows
# ============================================================

goalie_players_created = 0
goalie_players_failed = []

for _, p in all_goalie_df.iterrows():

    player_id = int(p["playerId"])

    # Check whether player already exists
    existing = conn.execute(
        """
        SELECT player_id
        FROM players
        WHERE player_id = ?
        """,
        (player_id,)
    ).fetchone()

    if existing:
        continue

    try:
        team_abbrevs = (p.get("teamAbbrevs") or "").strip()

        team_abbrev = (
            team_abbrevs.split(",")[0].strip()
            if team_abbrevs
            else None
        )

        if not team_abbrev:
            raise ValueError(
                f"No team abbreviation for player {player_id}"
            )

        team_id = nhl.get_team_id(
            conn,
            team_abbrev
        )

        if team_id is None:
            raise ValueError(
                f"Could not resolve team {team_abbrev}"
            )

        # Fetch full player profile
        profile = nhl.get_json(
            f"{nhl.BASE_WEB}/player/{player_id}/landing"
        )

        player_row = {
            "player_id": profile["playerId"],
            "team_id": team_id,
            "first_name": profile["firstName"]["default"],
            "last_name": profile["lastName"]["default"],
            "position": profile.get("position"),
            "jersey_number": profile.get("sweaterNumber"),
            "birth_date": profile.get("birthDate"),
            "birth_country": profile.get("birthCountry"),
            "height_cm": profile.get("heightInCentimeters"),
            "weight_kg": profile.get("weightInKilograms"),
            "shoots_catches": profile.get("shootsCatches"),
            "headshot_url": profile.get("headshot"),
        }

        nhl.insert_ignore(
            conn,
            "players",
            player_row
        )

        goalie_players_created += 1

        print(
            f"Created player: "
            f"{player_id} - "
            f"{player_row['first_name']} "
            f"{player_row['last_name']} "
            f"({team_abbrev})"
        )

    except Exception as e:

        goalie_players_failed.append({
            "player_id": player_id,
            "reason": str(e)
        })

        print(
            f"FAILED player {player_id}: "
            f"{type(e).__name__}: {e}"
        )


conn.commit()

print("\nGoalie player preparation finished.")
print("Players created:", goalie_players_created)
print("Players failed:", len(goalie_players_failed))

Created player: 8478971 - Connor Ingram (EDM)
Created player: 8473503 - James Reimer (OTT)
Created player: 8471734 - Jonathan Quick (NYR)
Created player: 8475852 - Petr Mrazek (ANA)
Created player: 8483668 - Nikke Kokko (SEA)
Created player: 8477484 - Spencer Martin (NYR)
Created player: 8484910 - Victor Ostman (SEA)
Created player: 8477405 - Marcus Hogberg (NYI)
Created player: 8482411 - Hunter Shepard (OTT)
Created player: 8483746 - Vyacheslav Buteyets (ANA)
Created player: 8476899 - Matt Murray (SEA)

Goalie player preparation finished.
Players created: 11
Players failed: 0


In [48]:
missing_after = []

for player_id in sorted(goalie_player_ids):

    exists = conn.execute(
        """
        SELECT 1
        FROM players
        WHERE player_id = ?
        """,
        (player_id,)
    ).fetchone()

    if not exists:
        missing_after.append(player_id)

print("Goalie players still missing:", len(missing_after))

if missing_after:
    print(missing_after)
else:
    print("All 98 goalie players now exist in players table.")

print(
    "\nTotal players:",
    pd.read_sql(
        "SELECT COUNT(*) AS n FROM players",
        conn
    ).iloc[0]["n"]
)

Goalie players still missing: 0
All 98 goalie players now exist in players table.

Total players: 1680


## Part C — `goalie_season_stats`

In [ ]:
sample_g = nhl.fetch_goalie_summary(start=0, limit=5)
sample_g['data'][0]


In [49]:
# start, limit = 0, 100
# g_loaded = 0

# while True:
#     page = nhl.fetch_goalie_summary(start=start, limit=limit)
#     if not page or not page.get('data'):
#         break
#     for p in page['data']:
#         team_abbrev = (p.get('teamAbbrevs') or '').split(',')[0] or None
#         team_id = nhl.get_team_id(conn, team_abbrev) if team_abbrev else None
#         row = {
#             'player_id': p['playerId'],
#             'season': str(p.get('seasonId')),
#             'team_id': team_id,
#             'games_played': p.get('gamesPlayed'),
#             'wins': p.get('wins'),
#             'losses': p.get('losses'),
#             'ot_losses': p.get('otLosses'),
#             'save_pct': p.get('savePctg'),
#             'goals_against_avg': p.get('goalsAgainstAverage'),
#             'shutouts': p.get('shutouts'),
#             'saves': p.get('saves'),
#         }
#         nhl.insert_ignore(conn, 'goalie_season_stats', row)
#         g_loaded += 1
#     if len(page['data']) < limit:
#         break
#     start += limit

# conn.commit()
# print(f'{g_loaded} goalie_season_stats rows inserted')


# ============================================================
# Load goalie season statistics
# One row per (player_id, season)
# ============================================================

import time

BATCH_SIZE = 100
start = 0
limit = 100

g_loaded = 0
g_failed = []

while True:

    page = nhl.fetch_goalie_summary(
        start=start,
        limit=limit
    )

    if not page or not page.get("data"):
        break

    for p in page["data"]:

        try:
            player_id = p["playerId"]
            season = str(p["seasonId"])

            # A player may have played for multiple teams
            # during the season, e.g.:
            #   EDM,PIT
            #
            # The API provides one season-total record.
            # Our existing schema has one team_id column,
            # so we use the first listed team as the
            # representative team for this season row.
            team_abbrevs = (
                p.get("teamAbbrevs") or ""
            ).strip()

            team_abbrev = (
                team_abbrevs.split(",")[0].strip()
                if team_abbrevs
                else None
            )

            team_id = (
                nhl.get_team_id(
                    conn,
                    team_abbrev
                )
                if team_abbrev
                else None
            )

            if team_abbrev and team_id is None:
                raise ValueError(
                    f"Team abbreviation {team_abbrev} "
                    f"not found for player {player_id}"
                )

            row = {
                "player_id": player_id,
                "season": season,
                "team_id": team_id,
                "games_played": p.get("gamesPlayed"),
                "wins": p.get("wins"),
                "losses": p.get("losses"),
                "ot_losses": p.get("otLosses"),
                "save_pct": p.get("savePct"),
                "goals_against_avg": p.get(
                    "goalsAgainstAverage"
                ),
                "shutouts": p.get("shutouts"),
                "saves": p.get("saves"),
            }

            nhl.insert_ignore(
                conn,
                "goalie_season_stats",
                row
            )

            g_loaded += 1

        except Exception as e:

            g_failed.append({
                "player_id": p.get("playerId"),
                "season": p.get("seasonId"),
                "reason": str(e)
            })

            print(
                f"FAILED goalie {p.get('playerId')}: "
                f"{type(e).__name__}: {e}"
            )

    # Commit after each page
    conn.commit()

    print(
        f"Progress: {g_loaded}/{page.get('total', '?')} "
        f"goalie records processed"
    )

    if len(page["data"]) < limit:
        break

    start += limit

    time.sleep(0.05)


print("\nGoalie season-stat load finished.")
print("Records processed:", g_loaded)
print("Failed records:", len(g_failed))

Progress: 98/98 goalie records processed

Goalie season-stat load finished.
Records processed: 98
Failed records: 0


In [50]:
print(
    pd.read_sql(
        """
        SELECT COUNT(*) AS n
        FROM goalie_season_stats
        """,
        conn
    )
)

    n
0  98


In [51]:
# ============================================================
# Validate goalie season statistics
# ============================================================

goalie_count = pd.read_sql(
    """
    SELECT COUNT(*) AS n
    FROM goalie_season_stats
    """,
    conn
).iloc[0]["n"]

duplicate_count = pd.read_sql(
    """
    SELECT COUNT(*) AS n
    FROM (
        SELECT player_id, season
        FROM goalie_season_stats
        GROUP BY player_id, season
        HAVING COUNT(*) > 1
    )
    """,
    conn
).iloc[0]["n"]

orphan_players = pd.read_sql(
    """
    SELECT COUNT(*) AS n
    FROM goalie_season_stats g
    LEFT JOIN players p
        ON g.player_id = p.player_id
    WHERE p.player_id IS NULL
    """,
    conn
).iloc[0]["n"]

orphan_teams = pd.read_sql(
    """
    SELECT COUNT(*) AS n
    FROM goalie_season_stats g
    LEFT JOIN teams t
        ON g.team_id = t.team_id
    WHERE g.team_id IS NOT NULL
      AND t.team_id IS NULL
    """,
    conn
).iloc[0]["n"]


print("Goalie season-stat rows:", goalie_count)
print("Duplicate player-season groups:", duplicate_count)
print("Orphan player references:", orphan_players)
print("Orphan team references:", orphan_teams)

Goalie season-stat rows: 98
Duplicate player-season groups: 0
Orphan player references: 0
Orphan team references: 0


In [52]:
# ============================================================
# NHL Analytics Hub - Database Overview
# Final data-loading checkpoint
# ============================================================

tables = [
    "teams",
    "standings",
    "players",
    "games",
    "game_stats",
    "skater_season_stats",
    "goalie_season_stats"
]

print("=" * 60)
print("NHL ANALYTICS HUB - DATABASE OVERVIEW")
print("=" * 60)

for table in tables:

    count = pd.read_sql(
        f"SELECT COUNT(*) AS n FROM {table}",
        conn
    ).iloc[0]["n"]

    print(f"{table:<25} {count:>8}")

print("=" * 60)

NHL ANALYTICS HUB - DATABASE OVERVIEW
teams                           32
standings                       32
players                       1680
games                         1498
game_stats                   53927
skater_season_stats            940
goalie_season_stats             98


In [53]:
print("=" * 70)
print("NHL ANALYTICS HUB - DATABASE INTEGRITY VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. ROW COUNTS
# ------------------------------------------------------------

tables = [
    "teams",
    "standings",
    "players",
    "games",
    "game_stats",
    "skater_season_stats",
    "goalie_season_stats"
]

print("\n[1] ROW COUNTS")
for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS n FROM {table}",
        conn
    ).iloc[0]["n"]

    print(f"{table:<25} {count:>8}")


# ------------------------------------------------------------
# 2. PRIMARY-KEY / NATURAL-KEY DUPLICATES
# ------------------------------------------------------------

print("\n[2] DUPLICATE KEY CHECKS")

checks = {
    "Duplicate team_id": """
        SELECT team_id, COUNT(*) AS n
        FROM teams
        GROUP BY team_id
        HAVING COUNT(*) > 1
    """,

    "Duplicate player_id": """
        SELECT player_id, COUNT(*) AS n
        FROM players
        GROUP BY player_id
        HAVING COUNT(*) > 1
    """,

    "Duplicate game_id": """
        SELECT game_id, COUNT(*) AS n
        FROM games
        GROUP BY game_id
        HAVING COUNT(*) > 1
    """,

    "Duplicate game/player": """
        SELECT game_id, player_id, COUNT(*) AS n
        FROM game_stats
        GROUP BY game_id, player_id
        HAVING COUNT(*) > 1
    """,

    "Duplicate skater player/season": """
        SELECT player_id, season, COUNT(*) AS n
        FROM skater_season_stats
        GROUP BY player_id, season
        HAVING COUNT(*) > 1
    """,

    "Duplicate goalie player/season": """
        SELECT player_id, season, COUNT(*) AS n
        FROM goalie_season_stats
        GROUP BY player_id, season
        HAVING COUNT(*) > 1
    """
}

for name, query in checks.items():
    result = pd.read_sql(query, conn)
    status = "PASS" if result.empty else "FAIL"
    print(f"{name:<35} {status}")


# ------------------------------------------------------------
# 3. FOREIGN-KEY INTEGRITY
# ------------------------------------------------------------

print("\n[3] FOREIGN-KEY CHECK")

fk_result = pd.read_sql(
    "PRAGMA foreign_key_check",
    conn
)

print(
    "Foreign-key violations:",
    len(fk_result)
)

if fk_result.empty:
    print("Status: PASS")
else:
    print("Status: FAIL")
    display(fk_result.head(20))


# ------------------------------------------------------------
# 4. ORPHAN REFERENCE CHECKS
# ------------------------------------------------------------

print("\n[4] ORPHAN REFERENCE CHECKS")

orphan_checks = {
    "Orphan standings teams": """
        SELECT COUNT(*) AS n
        FROM standings s
        LEFT JOIN teams t
            ON s.team_id = t.team_id
        WHERE t.team_id IS NULL
    """,

    "Orphan player teams": """
        SELECT COUNT(*) AS n
        FROM players p
        LEFT JOIN teams t
            ON p.team_id = t.team_id
        WHERE t.team_id IS NULL
    """,

    "Orphan game home teams": """
        SELECT COUNT(*) AS n
        FROM games g
        LEFT JOIN teams t
            ON g.home_team_id = t.team_id
        WHERE t.team_id IS NULL
    """,

    "Orphan game away teams": """
        SELECT COUNT(*) AS n
        FROM games g
        LEFT JOIN teams t
            ON g.away_team_id = t.team_id
        WHERE t.team_id IS NULL
    """,

    "Orphan game_stats players": """
        SELECT COUNT(*) AS n
        FROM game_stats gs
        LEFT JOIN players p
            ON gs.player_id = p.player_id
        WHERE p.player_id IS NULL
    """,

    "Orphan game_stats games": """
        SELECT COUNT(*) AS n
        FROM game_stats gs
        LEFT JOIN games g
            ON gs.game_id = g.game_id
        WHERE g.game_id IS NULL
    """,

    "Orphan game_stats teams": """
        SELECT COUNT(*) AS n
        FROM game_stats gs
        LEFT JOIN teams t
            ON gs.team_id = t.team_id
        WHERE t.team_id IS NULL
    """
}

for name, query in orphan_checks.items():
    count = pd.read_sql(query, conn).iloc[0]["n"]
    status = "PASS" if count == 0 else "FAIL"
    print(f"{name:<35} {status} ({count})")


# ------------------------------------------------------------
# 5. REQUIRED FIELD NULL CHECKS
# ------------------------------------------------------------

print("\n[5] REQUIRED FIELD NULL CHECKS")

null_checks = {
    "Teams: team_id": """
        SELECT COUNT(*) AS n
        FROM teams
        WHERE team_id IS NULL
    """,

    "Teams: abbreviation": """
        SELECT COUNT(*) AS n
        FROM teams
        WHERE team_abbrev IS NULL
    """,

    "Players: player_id": """
        SELECT COUNT(*) AS n
        FROM players
        WHERE player_id IS NULL
    """,

    "Players: team_id": """
        SELECT COUNT(*) AS n
        FROM players
        WHERE team_id IS NULL
    """,

    "Games: game_id": """
        SELECT COUNT(*) AS n
        FROM games
        WHERE game_id IS NULL
    """,

    "Games: home_team_id": """
        SELECT COUNT(*) AS n
        FROM games
        WHERE home_team_id IS NULL
    """,

    "Games: away_team_id": """
        SELECT COUNT(*) AS n
        FROM games
        WHERE away_team_id IS NULL
    """,

    "Game stats: game_id": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE game_id IS NULL
    """,

    "Game stats: player_id": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE player_id IS NULL
    """,

    "Game stats: team_id": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE team_id IS NULL
    """
}

for name, query in null_checks.items():
    count = pd.read_sql(query, conn).iloc[0]["n"]
    status = "PASS" if count == 0 else "FAIL"
    print(f"{name:<35} {status} ({count})")


# ------------------------------------------------------------
# 6. GAME TYPE DISTRIBUTION
# ------------------------------------------------------------

print("\n[6] GAME TYPE DISTRIBUTION")

game_types = pd.read_sql(
    """
    SELECT
        game_type,
        COUNT(*) AS games
    FROM games
    GROUP BY game_type
    ORDER BY game_type
    """,
    conn
)

display(game_types)


# ------------------------------------------------------------
# 7. COMPLETED GAMES WITHOUT PLAYER STATS
# ------------------------------------------------------------

print("\n[7] COMPLETED GAMES WITHOUT PLAYER STATS")

missing_stats = pd.read_sql(
    """
    SELECT
        g.game_id,
        g.game_type,
        g.game_state
    FROM games g
    LEFT JOIN game_stats gs
        ON g.game_id = gs.game_id
    WHERE g.game_state IN ('OFF', 'FINAL')
    GROUP BY g.game_id
    HAVING COUNT(gs.player_id) = 0
    """,
    conn
)

print(
    "Completed games without game_stats:",
    len(missing_stats)
)

print(
    "Status:",
    "PASS" if missing_stats.empty else "FAIL"
)

if not missing_stats.empty:
    display(missing_stats.head(20))


# ------------------------------------------------------------
# 8. TEAMS WITHOUT STANDINGS
# ------------------------------------------------------------

print("\n[8] TEAMS WITHOUT STANDINGS")

teams_without_standings = pd.read_sql(
    """
    SELECT
        t.team_id,
        t.team_abbrev,
        t.team_name
    FROM teams t
    LEFT JOIN standings s
        ON t.team_id = s.team_id
    WHERE s.team_id IS NULL
    """,
    conn
)

print(
    "Teams without standings:",
    len(teams_without_standings)
)

print(
    "Status:",
    "PASS" if teams_without_standings.empty else "FAIL"
)

if not teams_without_standings.empty:
    display(teams_without_standings)


# ------------------------------------------------------------
# 9. BASIC VALUE SANITY CHECKS
# ------------------------------------------------------------

print("\n[9] BASIC VALUE SANITY CHECKS")

sanity_checks = {
    "Games with negative scores": """
        SELECT COUNT(*) AS n
        FROM games
        WHERE home_score < 0
           OR away_score < 0
    """,

    "Game stats negative goals": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE goals < 0
    """,

    "Game stats negative assists": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE assists < 0
    """,

    "Game stats negative points": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE points < 0
    """,

    "Game stats negative shots": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE shots_on_goal < 0
    """,

    "Game stats negative penalty minutes": """
        SELECT COUNT(*) AS n
        FROM game_stats
        WHERE penalty_min < 0
    """
}

for name, query in sanity_checks.items():
    count = pd.read_sql(query, conn).iloc[0]["n"]
    status = "PASS" if count == 0 else "CHECK"
    print(f"{name:<35} {status} ({count})")


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATABASE INTEGRITY VALIDATION COMPLETE")
print("=" * 70)

NHL ANALYTICS HUB - DATABASE INTEGRITY VALIDATION

[1] ROW COUNTS
teams                           32
standings                       32
players                       1680
games                         1498
game_stats                   53927
skater_season_stats            940
goalie_season_stats             98

[2] DUPLICATE KEY CHECKS
Duplicate team_id                   PASS
Duplicate player_id                 PASS
Duplicate game_id                   PASS
Duplicate game/player               PASS
Duplicate skater player/season      PASS
Duplicate goalie player/season      PASS

[3] FOREIGN-KEY CHECK
Foreign-key violations: 0
Status: PASS

[4] ORPHAN REFERENCE CHECKS
Orphan standings teams              PASS (0)
Orphan player teams                 PASS (0)
Orphan game home teams              PASS (0)
Orphan game away teams              PASS (0)
Orphan game_stats players           PASS (0)
Orphan game_stats games             PASS (0)
Orphan game_stats teams             PASS (0)

[5] REQUIR

,game_type,games
0,1,104
1,2,1312
2,3,82



[7] COMPLETED GAMES WITHOUT PLAYER STATS
Completed games without game_stats: 0
Status: PASS

[8] TEAMS WITHOUT STANDINGS
Teams without standings: 0
Status: PASS

[9] BASIC VALUE SANITY CHECKS
Games with negative scores          PASS (0)
Game stats negative goals           PASS (0)
Game stats negative assists         PASS (0)
Game stats negative points          PASS (0)
Game stats negative shots           PASS (0)
Game stats negative penalty minutes PASS (0)

DATABASE INTEGRITY VALIDATION COMPLETE


In [ ]:
for t in ['game_stats', 'skater_season_stats', 'goalie_season_stats']:
    print(t, pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0]['n'])
conn.close()
